# Day 7 · Transformer assignment

**Core work: 30–40 minutes, after setup.** Work alone or in pairs. This notebook runs independently of the demonstration; it does not rely on variables from another kernel. Use the [guided lab](../Transformers_what_can_they_do_.ipynb) as a reference.

**Submit:** this notebook with your completed code, your own outputs and short written answers. Required evidence is a task/model/architecture table, token-count comparison, sentiment stress test, two bounded sampled continuations, a greedy comparison and one limitation. Exact generated text need not match another learner's.

Cells marked `TODO` are deliberately incomplete. Running the starter notebook prints reminders rather than pretending the exercises are solved. Short hints follow each task. Instructor solutions are in a separate file and are not needed to begin.

## Setup - do this before the session

Use a **fresh Python 3.12 environment** with **PyTorch 2.6.0 CPU** and **Transformers 4.57.1**. These are the course pins, not a claim that they are the newest releases. Do not upgrade to v5 during the session: some older pipeline tasks, including summarization and translation, were removed from that API. A v5 migration needs different task code and a new smoke test. [Versioned pipeline documentation](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines) · [Current translation model instructions](https://huggingface.co/Helsinki-NLP/opus-mt-fr-en)

The first model load downloads public weights and tokenizer files. The three core weight files total about **0.89 GB**, excluding packages and cache overhead. Allow **at least 3 GB of free disk space** for the core environment/cache. A recent laptop with **4 CPU cores and 8 GB RAM** is the planning assumption. Budget **5–20 minutes for first downloads and setup**, and roughly **1–5 minutes of computation after caching**; these are planning estimates, not benchmarks or guarantees. Network speed, package installation and hardware can change them substantially.

Examples use invented, non-sensitive text. The code runs model inference locally; downloading weights requires a Hub connection. Do not paste client material into a public Hub widget. No Hugging Face login or paid inference API is required for these public checkpoints.

In [ ]:
# Optional one-time installation in a fresh notebook kernel, then restart it:
# %pip install "torch==2.6.0" --index-url https://download.pytorch.org/whl/cpu
# %pip install "transformers==4.57.1"
# On macOS, use this torch install line instead of the CPU-index line:
# %pip install "torch==2.6.0"

In [ ]:
import os
import platform
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version

OFFLINE = False  # Change before imports; restart the kernel after changing it.
if OFFLINE:
    os.environ['HF_HUB_OFFLINE'] = '1'
os.environ.setdefault('HF_HOME', str(Path.home() / '.cache' / 'huggingface'))
LOCAL_FILES_ONLY = OFFLINE or os.environ.get('HF_HUB_OFFLINE') == '1'

def require_versions(required):
    for package, expected in required.items():
        try:
            actual = version(package)
        except PackageNotFoundError as exc:
            raise RuntimeError(f'Install {package}=={expected} in this kernel, then restart it.') from exc
        print(f'{package}: {actual}')
        if actual.split('+')[0] != expected:
            raise RuntimeError(f'This lab pins {package}=={expected}; use the course environment and restart.')

require_versions({'torch': '2.6.0', 'transformers': '4.57.1'})
import torch
from transformers import AutoTokenizer, pipeline, set_seed

torch.set_num_threads(min(4, os.cpu_count() or 1))
SEED = 42
set_seed(SEED)
print('Python:', platform.python_version())
print('Device: CPU; threads:', torch.get_num_threads())
print('HF_HOME:', os.environ['HF_HOME'])
print('Local cache only:', LOCAL_FILES_ONLY)

In [ ]:
MODEL_SPECS = {'sentiment': {'task': 'sentiment-analysis',
               'model': 'distilbert/distilbert-base-uncased-finetuned-sst-2-english',
               'revision': '714eb0fa89d2f80546fda750413ed43d93601a13'},
 'mask': {'task': 'fill-mask',
          'model': 'distilbert/distilbert-base-uncased',
          'revision': '12040accade4e8a0f71eabdb258fecc2e7e948be'},
 'generation': {'task': 'text-generation',
                'model': 'distilbert/distilgpt2',
                'revision': '2290a62682d06624634c1f46a6ad5be0f47f38aa'}}

def load_pipeline(name, **task_options):
    spec = MODEL_SPECS[name]
    tokenizer = AutoTokenizer.from_pretrained(
        spec['model'], revision=spec['revision'],
        local_files_only=LOCAL_FILES_ONLY, trust_remote_code=False,
    )
    result = pipeline(
        task=spec['task'], model=spec['model'], tokenizer=tokenizer,
        revision=spec['revision'], framework='pt', device=-1,
        trust_remote_code=False,
        model_kwargs={'use_safetensors': True, 'local_files_only': LOCAL_FILES_ONLY},
        **task_options,
    )
    print(f"{name}: {spec['model']} @ {spec['revision'][:7]}")
    print(type(result.model).__name__, '/', type(result.tokenizer).__name__)
    return result

## Task 1 · Match the parts (5 minutes)

Complete the table. Use the three checkpoints in `MODEL_SPECS`. In the last column, describe the result's meaning rather than just its Python type.

| Task | Checkpoint | Architecture family | What the output means |
|---|---|---|---|
| Sentiment | TODO | TODO | TODO |
| Fill-mask | TODO | TODO | TODO |
| Text generation | TODO | TODO | TODO |

**Explain in 2–3 sentences:** why does selecting a base BERT-like encoder not automatically give us a trained sentiment classifier? What must match between a tokenizer and a checkpoint?

**Success:** all three mappings are complete; your explanation distinguishes architecture, learned weights, output head and tokenizer. **Hint:** read the model card's task and training description; `pipeline` is a wrapper, not an architecture.

**Your answer:** TODO

## Task 2 · Make tokens visible (7 minutes)

Load the mask pipeline in the next cell. For `token_text`, compute a whitespace word count, token IDs **without special tokens**, and a token count. Print the token strings too. Then construct one sentence using `unmasker.tokenizer.mask_token` and request its top three completions.

Change one context word and run the mask prediction again. Record the candidate token and score that changed most noticeably. Do not guess a prediction before running it.

**Success:** you show both counts and the actual IDs/token strings; two inputs each contain exactly one valid mask; each result has three candidates. Explain why top-three scores need not sum to one.

**Hint:** `tokenizer(text, add_special_tokens=False)` returns a dictionary containing `input_ids`; `convert_ids_to_tokens` makes those IDs readable. The tokenizer exposes `mask_token`.

In [ ]:
unmasker = load_pipeline('mask')
token_text = "Re-tokenization isn't always one word per token."
word_count = None      # TODO
token_ids = None       # TODO
token_count = None     # TODO
masked_text = None     # TODO: use the tokenizer's mask marker
mask_results = None    # TODO: call unmasker for three candidates

if any(value is None for value in [word_count, token_ids, token_count, masked_text, mask_results]):
    print('Complete Task 2 TODOs, then rerun this cell.')
else:
    print('Words:', word_count, 'Tokens:', token_count)
    print('Token IDs:', token_ids)
    print('Token strings:', unmasker.tokenizer.convert_ids_to_tokens(token_ids))
    print(mask_results)
    assert masked_text.count(unmasker.tokenizer.mask_token) == 1
    assert len(mask_results) == 3
    assert token_count == len(token_ids)

In [ ]:
changed_masked_text = None  # TODO: change one context word
changed_mask_results = None # TODO: rerun the same model with top_k=3
if changed_mask_results is None:
    print('Complete the second input and compare its observed ranking.')
else:
    print(changed_masked_text, changed_mask_results)

**Your interpretation:** TODO - include the two counts, one changed candidate and an explanation of the scores.

## Task 3 · Stress-test a classifier (7 minutes)

Before running the model, label the four examples yourself as positive, negative, neutral or ambiguous. Then call the sentiment pipeline on the entire list, with input truncation enabled and a 128-token input cap. Compare your assessment with the model's available labels.

**Success:** four observed predictions with scores; one disagreement or ambiguous case discussed; one sentence explaining why a high score does not validate use on client tickets. If you agree with all four outputs, add an invented neutral or mixed-sentiment example and discuss the forced binary choice.

**Hint:** this checkpoint has no neutral output class. Inspect `sentiment.model.config.id2label`. A tiny hand-picked sample illustrates a failure mode; it is not a benchmark.

In [ ]:
sentiment = load_pipeline('sentiment')
examples = [
    'The explanation was clear and useful.',
    'The application crashes every time I open it.',
    "The meeting starts at two o'clock.",
    'I loved the tutor, but the exercises were frustrating.',
]
my_labels = []              # TODO: four judgments before model inference
student_predictions = None # TODO: one pipeline call on examples
print('Available model labels:', sentiment.model.config.id2label)
if student_predictions is None or len(my_labels) != len(examples):
    print('Complete your judgments and the pipeline call.')
else:
    assert len(student_predictions) == len(examples)
    for text, human, model_result in zip(examples, my_labels, student_predictions):
        print({'text': text, 'my_judgment': human, 'model': model_result})

**Your interpretation:** TODO - identify an ambiguous or failed case, and propose one concrete check before applying this checkpoint to a different domain.

## Task 4 · Bound generation and compare decoding (10 minutes)

Load the generation model and run the provided reporting helper. Write a call that generates **two sampled continuations with at most 15 new tokens each**. Use a seed of 42, one beam and deliberate sampling parameters. Request token IDs with `return_tensors=True`; pass the tokenizer's EOS ID as `pad_token_id` for this model.

Then generate **one greedy continuation** with the same prompt and token cap. Print all three continuations with both token counts and whitespace word counts.

**Success:** two sampled results, one greedy result, and each count at most 15. Your explanation says “maximum new tokens,” allows early stopping and does not require exactly 15 words, two complete sentences or different sampled texts.

**Hint:** inspect `max_new_tokens`, `num_return_sequences` and `do_sample`. Greedy decoding with one beam uses one return sequence. Tune `temperature` and `top_p` for sampling. For the greedy call, use neutral `temperature=1.0` to reset this v4 pipeline's inherited 0.7 default and avoid an ignored-parameter warning; it has no sampling effect when `do_sample=False`.

In [ ]:
def describe_generations(results, prompt, tokenizer):
    """Count original generated IDs, rather than re-tokenizing decoded text."""
    prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
    rows = []
    for result in results:
        all_ids = result['generated_token_ids']
        assert all_ids[:len(prompt_ids)] == prompt_ids
        added_ids = all_ids[len(prompt_ids):]
        # DistilGPT2 uses EOS as padding here; ignore padding after the first EOS.
        if tokenizer.eos_token_id in added_ids:
            added_ids = added_ids[:added_ids.index(tokenizer.eos_token_id) + 1]
        continuation = tokenizer.decode(added_ids, skip_special_tokens=True)
        rows.append({
            'continuation': continuation,
            'new_tokens_including_eos': len(added_ids),
            'whitespace_words': len(continuation.split()),
        })
    return rows

In [ ]:
generator = load_pipeline('generation')
prompt = 'To prepare a useful training workshop, we should'
set_seed(SEED)
sampled_results = None  # TODO: two sampled continuations, return token IDs
greedy_results = None   # TODO: one greedy continuation, same token cap

In [ ]:
if sampled_results is None or greedy_results is None:
    print('Complete both generation calls, then rerun this check.')
else:
    sampled_rows = describe_generations(sampled_results, prompt, generator.tokenizer)
    greedy_rows = describe_generations(greedy_results, prompt, generator.tokenizer)
    assert len(sampled_rows) == 2
    assert len(greedy_rows) == 1
    assert all(1 <= row['new_tokens_including_eos'] <= 15 for row in sampled_rows + greedy_rows)
    print('Sampled:', sampled_rows)
    print('Greedy:', greedy_rows)

**Your interpretation:** TODO - compare the observed outputs and explain (a) tokens versus words, (b) sampling versus greedy, and (c) why neither method guarantees factuality. What would an application need to do if it genuinely required exactly 15 whitespace-delimited words?

## Task 5 · Make a model decision (5 minutes)

A team wants to use one of these checkpoints on a new task. Choose **one** scenario and write 4–6 sentences:

- Route short English customer-support messages to three departments.
- Suggest a continuation for public training copy.
- Extract named organizations from public event announcements.

Name a suitable task and model family, identify a mismatch with at least one of the core checkpoints, give one test case, and state one evaluation measure or review criterion. Also identify a case the system should hand to a human. You do not need to download another model.

**Success:** your recommendation follows the task and evidence; it does not use a sentiment score as routing accuracy or fluency as factual correctness.

**Your answer:** TODO

## Optional investigation - choose one (10–15 minutes)

These preserve the original exploration topics. A model-card review is enough for the first pass; download only after checking size and language support. If you run a model, use an explicit model ID and record its commit revision, task, tokenizer, library version and actual output. If you do not run it, state **not executed** and explain the planned test.

| Option | Action | Evidence to submit |
|---|---|---|
| Another language | Find a text-generation model whose card supports a language you know. | Model URL, language, architecture, task, weight size, limitation and one bounded test prompt. |
| Cased BERT | Inspect [google-bert/bert-base-cased](https://huggingface.co/google-bert/bert-base-cased). Identify its mask marker and contrast cased/uncased tokenization. | The marker and source; if executed, the top three observed candidates for one input. Do not copy a prediction from a widget. |
| POS tagging | Inspect [vblagoje/bert-english-uncased-finetuned-pos](https://huggingface.co/vblagoje/bert-english-uncased-finetuned-pos) or another English POS checkpoint. | Task, at least three tag meanings, and why those labels differ from NER. Note any missing license/training information in the card. |
| Translation | Choose one directed language pair and inspect a suitable model card. | Source/target language, tokenizer requirements, model URL and a plan to check numbers, names and meaning. |
| Original advanced demos | Enable one optional section in the guided lab: zero-shot, NER, QA or summary. | Your observed result and one limitation demonstrated by changing the input. |

**Your investigation and execution status:** TODO

## Submission check

- Replace every required `TODO` with your own code or interpretation.
- Restart and run the notebook in order using the pinned kernel. Keep the printed versions.
- Keep your observed outputs; do not invent a plausible result for an unexecuted cell.
- Explain one failure or ambiguity even when the code runs successfully.
- Mark the optional investigation as executed or not executed.

**Assessment:** task/model mapping (2 points), token and mask evidence (2), sentiment interpretation (2), bounded generation and decoding comparison (3), model decision and limitation (1). Total: **10 points**. Optional work is for discussion, not required to achieve 10/10.

## Sources and attribution

The original notebook follows the Hugging Face course chapter [Transformers, what can they do?](https://huggingface.co/learn/llm-course/chapter1/3). This revision uses new classroom examples and adds setup, model selection, bounded generation, exercises and interpretation. This is **source attribution**, not a license grant; the course source and each model retain their own applicable licenses. Model weights are downloaded separately and are not distributed in this notebook.

Primary references checked for this revision:

- [Transformers 4.57.1 pipeline API](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines), including tasks, NER aggregation and output formats.
- [Transformers 4.57.1 generation parameters](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/text_generation), including sampling and token limits.
- [Installation, caching and offline mode](https://huggingface.co/docs/transformers/v4.57.1/en/installation).
- [Sentiment model card](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english) · [masked-language model card](https://huggingface.co/distilbert/distilbert-base-uncased) · [DistilGPT2 model card](https://huggingface.co/distilbert/distilgpt2).
- Weight-size evidence: [sentiment files](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english/tree/main), [mask-model files](https://huggingface.co/distilbert/distilbert-base-uncased/tree/main), [generation-model files](https://huggingface.co/distilbert/distilgpt2/tree/main). The figures are for one safetensors weight file, not the entire repository or runtime memory.

Model revision hashes in the code freeze the reviewed files. A fixed seed improves repeatability in a fixed environment; it does not guarantee bit-identical text across different hardware, library versions or decoding implementations. Inspect warnings; this notebook does not suppress them globally.